Anmerkungen:
- Kein Einkommen im Datensatz; Belastungsquote vorhanden, absolutes Niveau fehlt; Modellgüte ist dadurch nach oben begrenzt

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml

# Holt den Datensatz "credit-g" von OpenML (braucht Internet, einmalig ~2 Sek.)
daten = fetch_openml("credit-g", version=1, as_frame=True)
df = daten.frame          # .frame = die ganze Tabelle inkl. Zielspalte "class"
df['ausfall'] = df['class'].map({'good': 0, 'bad': 1}).astype(int)
df.head(5)

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class,ausfall
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,...,67,none,own,2,skilled,1,yes,yes,good,0
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,...,22,none,own,1,skilled,1,none,yes,bad,1
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,...,49,none,own,1,unskilled resident,2,none,yes,good,0
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,...,45,none,for free,1,skilled,2,none,yes,good,0
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,...,53,none,for free,2,skilled,2,none,yes,bad,1


K1:
Ich kalkuliere den Mittelwert der Spalte ausfall mit der funktion mean(). Das Ergebnis ist der Anteil, weil die Werte mit 1 zu einer Summe addiert werden und dann durch die Anzahl der Einträge geteilt wird (geht nur für 0 und 1 Ausprägungen). Ich bekomme eine Zahl zurück.
K2:
Über Checking_status mit groupby() gruppieren, Ausfallquoten und Anzahl der Checking_status Kategorie pro Gruppe über agg(mean und count) ermitteln.
Selektionseffekt. Die Tabelle zeigt nur genehmigte Kredite. Ohne-Konto-Kunden wurden strenger gefiltert, also stehen hier nur die guten davon. Das Merkmal „no checking" sagt also nicht „kein Konto = sicher", sondern „kein Konto UND trotzdem genehmigt = sicher". Ein Modell lernt genau diese Verzerrung mit — ein Punkt für die Deutung in Teil 2.
K3:
„Quoten aus Gruppen unter ~30 Fällen nicht deuten; bei der Modellierung kleine Zwecke zusammenlegen oder weglassen." Das wird in Teil 2 praktisch relevant, wenn purpose als Merkmal kodiert wird.
K4:
„Betrag und Laufzeit korreliert → Koeffizienten in Teil 2 nur gemeinsam deuten."
Korrelation 0,62 — deutlich, aber kein Gleichlauf. Dein Verdacht ist bestätigt, und die Zahl sagt zugleich: Die beiden Merkmale tragen noch je eigene Information, man muss keins davon wegwerfen. Erst ab etwa 0,8–0,9 würde man eins streichen oder die beiden zu einem Merkmal verdichten (etwa Rate = Betrag ÷ Laufzeit — die hätte sogar eine schöne Bedeutung).
K5:
Echter Befund, kein Rechenfehler: Mit der dokumentierten Kodierung fiele die Gruppe mit der niedrigsten Belastung (Stufe 4, unter 20 %) am häufigsten aus, die mit der höchsten (Stufe 1, ≥ 35 %) am seltensten — monoton, aber verkehrt herum.

Zwei Lesarten, und ich kann aus den Daten nicht entscheiden, welche gilt:

Die Dokumentation stimmt, der Effekt ist wieder ein Selektionseffekt wie bei „no checking": Wer mit 35 % Belastung genehmigt wird, war in allen anderen Punkten überdurchschnittlich gut. Dann sagt das Merkmal nichts über Belastung, sondern über die Strenge der Prüfung.
Die Dokumentation ist an dieser Stelle falsch oder gespiegelt (dieser Datensatz ist dafür bekannt, dass mehrere Attributbeschreibungen unsauber sind). Dann steigt die Quote mit der Belastung, wie du es erwarten würdest.

Was für beide Lesarten gilt: Der Effekt ist klein (0,25 → 0,33) und die Belastungsquote ohne absolutes Einkommen ein schwaches Merkmal — dein Befund von gestern. Für Teil 2: als Zahl 1–4 drinlassen, aber den Koeffizienten nicht fachlich deuten.

Summary:„installment_commitment - Richtung des Effekts unklar (Dokumentation vs. Daten), nicht deuten."

In [ ]:
#K1:
df.ausfall.mean()

In [ ]:
#K2:
df.groupby('checking_status')['ausfall'].agg(['mean', 'count']).sort_values('mean', ascending=False)

In [ ]:
#K3:
df.groupby('purpose')['ausfall'].agg(['mean', 'count']).sort_values('mean', ascending=False)

In [ ]:
#K4:
display(df.groupby('class')[['credit_amount','duration']].mean())
display(df[['credit_amount', 'duration']].corr())

In [ ]:
#K5:
df.groupby('installment_commitment')['ausfall'].agg(['mean', 'count']).sort_values('mean', ascending=False)